# Assignment Module 2: Aircraft Classification

The goal of this assignment is to implement a neural network that classifies images of 100 aircraft model variants from the [Fine-Grained Visual Classification of Aircraft (**FGVC-Aircraft**) dataset](https://www.robots.ox.ac.uk/~vgg/data/fgvc-aircraft/). The assignment is divided into two parts: first, you will be asked to implement your own neural network for image classification from scratch; then, you will fine-tune a pretrained network provided by PyTorch.

![](https:///raw.githubusercontent.com/CVLAB-Unibo/ipcv-assignment-2/master/fgvc_aircraft_variants.svg)

## 0. Setup

All data, caches and training runs live in a shared Google Drive folder, so that runs can be resumed after a disconnection and
plots/tables can be regenerated without retraining. Every run is saved in `RUNS_DIR/<AUTHOR>_<run name>/`.

The setup and dataset sections run fine on a **CPU runtime**; a GPU is only needed for training.

In [ ]:
# ----------------------------------------------------------------------------
AUTHOR     = "simone"                          # prefix of every run name (one per team member)
DRIVE_DIR  = "/content/drive/MyDrive/IPCV_A2"  # shared folder (shortcut in "My Drive")
DATA_ROOT  = "/content/data"                   # local disk of the Colab VM (fast)
SHORT_SIDE = 256                               # short side of the cached images, in pixels
BANNER_PX  = 20                                # height of the copyright banner at the bottom (see 1.3.3)
SEED       = 42
# ----------------------------------------------------------------------------
CACHE_DIR = f"{DRIVE_DIR}/cache"
RUNS_DIR  = f"{DRIVE_DIR}/runs"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json
import math
import os
import random
import shutil
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import FGVCAircraft
from torchvision.datasets.utils import download_url, extract_archive
from torchvision.transforms import v2
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm

for d in (DRIVE_DIR, CACHE_DIR, RUNS_DIR):
    os.makedirs(d, exist_ok=True)


def fix_random(seed: int) -> None:
    """Fix all the possible sources of randomness."""
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


fix_random(SEED)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device} ({GPU_NAME}) | CPU cores: {os.cpu_count()}")

## Dataset

Download and acces the dataset through its official [PyTorch `FGVCAircraft` class](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.FGVCAircraft.html) (by setting its constructor argument `annotation_level` to `'variant'`).

### 1.1 Download

The archive (~2.6 GB) is downloaded from Oxford only once and a copy is stored on Drive; in later sessions it is copied from Drive
to the local disk of the VM and extracted there, which is much faster.

In [ ]:
TAR_NAME = "fgvc-aircraft-2013b.tar.gz"
DATASET_URL = "https://www.robots.ox.ac.uk/~vgg/data/fgvc-aircraft/archives/" + TAR_NAME


def ensure_dataset(drive_dir, data_root) -> None:
    data_root, drive_dir = Path(data_root), Path(drive_dir)
    if (data_root / "fgvc-aircraft-2013b").is_dir():
        print("Dataset already extracted.")
        return
    data_root.mkdir(parents=True, exist_ok=True)
    drive_tar, local_tar = drive_dir / "data" / TAR_NAME, data_root / TAR_NAME
    t0 = time.time()
    if drive_tar.exists():
        print("Copying the archive from Drive...")
        shutil.copy(drive_tar, local_tar)
    else:
        print("Archive not on Drive: downloading it from Oxford (only once)...")
        download_url(DATASET_URL, str(data_root), filename=TAR_NAME)
        drive_tar.parent.mkdir(parents=True, exist_ok=True)
        print("Saving a copy on Drive...")
        shutil.copy(local_tar, drive_tar)
    print("Extracting...")
    extract_archive(str(local_tar), str(data_root))
    local_tar.unlink()
    print(f"Done in {time.time() - t0:.0f}s")


ensure_dataset(DRIVE_DIR, DATA_ROOT)

### 1.2 In-memory cache

The original images are large JPEGs: decoding and resizing them at every epoch would make the CPU, not the GPU, the bottleneck.
We therefore preprocess each split once — crop the copyright banner, resize so that the short side is `SHORT_SIDE`
(keeping the aspect ratio) — and keep the result in RAM as `uint8` tensors. The cache is saved on Drive and reused in later sessions.
Random augmentations are still applied on the fly at every epoch (Section 1.4).

In [ ]:
def _load_one(path, short_side, banner_px):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    if banner_px > 0:
        img = img.crop((0, 0, w, h - banner_px))
    img = TF.resize(img, short_side, antialias=True)
    return TF.pil_to_tensor(img), (w, h)


def build_or_load_cache(split, data_root, cache_dir, short_side, banner_px, num_threads=8) -> dict:
    cache_path = Path(cache_dir) / f"{split}_s{short_side}_b{banner_px}.pt"
    if cache_path.exists():
        t0 = time.time()
        cache = torch.load(cache_path)
        print(f"[{split}] loaded {len(cache['labels'])} images from cache in {time.time() - t0:.0f}s")
        return cache
    ds = FGVCAircraft(root=data_root, split=split, annotation_level="variant", download=False)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=num_threads) as ex:
        results = list(tqdm(ex.map(lambda p: _load_one(p, short_side, banner_px), ds._image_files),
                            total=len(ds), desc=split))
    cache = {"images": [r[0] for r in results], "labels": list(ds._labels),
             "orig_sizes": [r[1] for r in results], "classes": list(ds.classes)}
    print(f"[{split}] {len(ds)} images processed in {time.time() - t0:.0f}s")
    torch.save(cache, cache_path)
    return cache


caches = {s: build_or_load_cache(s, DATA_ROOT, CACHE_DIR, SHORT_SIDE, BANNER_PX) for s in ["train", "val", "test"]}
classes = caches["train"]["classes"]
num_classes = len(classes)
ram_gb = sum(im.numel() for c in caches.values() for im in c["images"]) / 1e9
print(f"{num_classes} classes | cache size in RAM: {ram_gb:.2f} GB")

### 1.3 Data exploration

#### 1.3.1 Splits and class balance

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3), sharey=True)
for a, s in zip(ax, ["train", "val", "test"]):
    cnt = Counter(caches[s]["labels"])
    counts = [cnt[i] for i in range(num_classes)]
    a.bar(range(num_classes), counts)
    a.set_title(f"{s}: {len(caches[s]['labels'])} images ({min(counts)}-{max(counts)} per class)")
    a.set_xlabel("class index")
ax[0].set_ylabel("# images")
plt.tight_layout()
plt.show()

In [ ]:
# Data-leakage check: the three splits must not share any image
files = {s: set(FGVCAircraft(DATA_ROOT, split=s, annotation_level="variant")._image_files) for s in ["train", "val", "test"]}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    print(f"{a} ∩ {b}: {len(files[a] & files[b])} shared images")

> *Observations:* the dataset is perfectly balanced (33–34 images per class in every split), so plain accuracy is a fair metric and no class re-weighting is needed. With only ~33 training images per class, overfitting is the main risk: regularization and data augmentation will be essential.

#### 1.3.2 Image size and aspect ratio

In [ ]:
sizes = np.array([s for c in caches.values() for s in c["orig_sizes"]])
w, h = sizes[:, 0], sizes[:, 1] - BANNER_PX
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(w, h, s=2, alpha=0.3)
ax[0].set_xlabel("width"); ax[0].set_ylabel("height (without banner)"); ax[0].set_title("Original image sizes")
ax[1].hist(w / h, bins=50)
ax[1].set_xlabel("aspect ratio (w / h)"); ax[1].set_title(f"Aspect ratio (median {np.median(w / h):.2f})")
plt.tight_layout()
plt.show()

> *Observations:* original images are large (800–1600 px wide), so downsampling to ~224 px discards fine details. The aspect ratio is tightly concentrated around 1.5 (median 1.52): a square center crop keeps only ~66% of the width and can cut the nose and tail of the aircraft (see 1.4), which motivates comparing different ways of obtaining the network input.

#### 1.3.3 Copyright banner

The bottom of each original image contains a copyright banner, which carries no information about the aircraft.
The red line marks the crop applied in the cache (`BANNER_PX` pixels from the bottom).

In [ ]:
raw = FGVCAircraft(DATA_ROOT, split="train", annotation_level="variant")
idxs = np.linspace(0, len(raw) - 1, 6).astype(int)
fig, ax = plt.subplots(2, 3, figsize=(15, 4))
for a, i in zip(ax.flat, idxs):
    img = np.array(raw[i][0])
    a.imshow(img[-80:])
    a.axhline(80 - BANNER_PX, color="red")
    a.set_title(f"{img.shape[1]}x{img.shape[0]}"); a.axis("off")
plt.suptitle("Bottom 80 pixels of some original images")
plt.tight_layout()
plt.show()

#### 1.3.4 Examples

In [ ]:
rng = np.random.default_rng(SEED)
train_labels = np.array(caches["train"]["labels"])
fig, ax = plt.subplots(3, 4, figsize=(16, 9))
for a, c in zip(ax.flat, rng.choice(num_classes, 12, replace=False)):
    i = rng.choice(np.where(train_labels == c)[0])
    a.imshow(caches["train"]["images"][i].permute(1, 2, 0))
    a.set_title(classes[c]); a.axis("off")
plt.tight_layout()
plt.show()

Variants of the same family (here: Boeing 737) differ only in small details — this is what makes the task *fine-grained*.

In [ ]:
family = [i for i, c in enumerate(classes) if c.startswith("737")][:6]
fig, ax = plt.subplots(1, len(family), figsize=(3 * len(family), 3), squeeze=False)
for a, c in zip(ax[0], family):
    i = np.where(train_labels == c)[0][0]
    a.imshow(caches["train"]["images"][i].permute(1, 2, 0))
    a.set_title(classes[c]); a.axis("off")
plt.tight_layout()
plt.show()

> *Observations:* aircraft are mostly seen from the side, centered and covering most of the image width. Variants of the same family (e.g. 737-200 … 737-700) are nearly identical, differing mainly in fuselage length and small details, so most errors are expected within families. The airline livery is a potential shortcut, since the same airline operates several variants and the same variant appears with many liveries. Some photos are grayscale, which supports the use of color augmentations.

### 1.4 Preprocessing, augmentation and data loaders

Images are normalized with the ImageNet statistics in every part of the assignment, so that the pretrained ResNet-18 of Part 2
receives inputs in the distribution it was trained on, and Part 2A can truly reuse the pipeline of Part 1 unchanged.

Since images are ~1.5 times wider than tall (Section 1.3.2), we compare three ways of obtaining the network input
(`resize_mode`):
- `"crop"`: resize the short side, then take a central square crop — may cut the nose and the tail of the aircraft;
- `"squash"`: resize the whole image to a square — keeps the whole aircraft, but compresses it horizontally;
- `"rect"`: resize the whole image to a rectangle with aspect ratio 1.5 (e.g. 192×288) — keeps the whole aircraft with almost
  no distortion, at a cost similar to 224×224 (55k vs 50k pixels). This works because networks ending with a global average
  pooling accept inputs of any spatial size.

At training time, `RandomResizedCrop` samples crops whose aspect ratio is centered on the one of the evaluation geometry,
so that aircraft have the same shape at training and test time (1 for `"crop"`, 1.5 for `"squash"` and `"rect"`).
The `"strong"` augmentation adds color jitter and random grayscale conversion, motivated by the livery shortcut and by the
presence of grayscale photos.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
DATA_ASPECT = 1.5  # median width / height of the dataset (Section 1.3.2)


def input_hw(cfg: dict):
    """(H, W) of the network input: img_size is an int (square) or [H, W] (rect)."""
    size = cfg.get("img_size", 224)
    return (size, size) if isinstance(size, int) else tuple(size)


def make_transforms(train: bool, cfg: dict):
    """cfg keys (defaults):
       img_size (224) | resize_mode ("crop" | "squash" | "rect") | aug ("none" | "basic" | "strong") | rrc_scale ((0.35, 1))
    """
    mode, aug = cfg.get("resize_mode", "crop"), cfg.get("aug", "basic")
    h, w = input_hw(cfg)
    if mode == "crop":
        assert h == w, "crop mode needs a square img_size"
        ops, ar = [v2.Resize(int(round(h * 256 / 224)), antialias=True), v2.CenterCrop(h)], 1.0
    elif mode == "squash":
        assert h == w, "squash mode needs a square img_size"
        ops, ar = [v2.Resize((h, w), antialias=True)], DATA_ASPECT
    elif mode == "rect":
        ops, ar = [v2.Resize((h, w), antialias=True)], w / h
    else:
        raise ValueError(mode)
    if train and aug != "none":
        ops = [v2.RandomResizedCrop((h, w), scale=tuple(cfg.get("rrc_scale", (0.35, 1.0))),
                                    ratio=(ar * 3 / 4, ar * 4 / 3), antialias=True),
               v2.RandomHorizontalFlip()]
        if aug == "strong":
            ops += [v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3), v2.RandomGrayscale(p=0.1)]
        elif aug != "basic":
            raise ValueError(aug)
    return v2.Compose(ops + [v2.ToDtype(torch.float32, scale=True), v2.Normalize(IMAGENET_MEAN, IMAGENET_STD)])


class CachedDataset(Dataset):
    def __init__(self, cache: dict, transform=None):
        self.images, self.labels, self.transform = cache["images"], cache["labels"], transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        x = self.images[i]
        return (self.transform(x) if self.transform else x), self.labels[i]


def make_loader(split: str, cfg: dict, train: bool, batch_size=None, seed=SEED):
    ds = CachedDataset(caches[split], make_transforms(train, cfg))
    g = torch.Generator(); g.manual_seed(seed)
    nw = os.cpu_count()
    return DataLoader(ds, batch_size=batch_size or cfg.get("batch_size", 64), shuffle=train, drop_last=train,
                      num_workers=nw, pin_memory=torch.cuda.is_available(), persistent_workers=nw > 0,
                      generator=g if train else None)


def denormalize(x: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor(IMAGENET_MEAN).view(-1, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(-1, 1, 1)
    return (x.cpu() * std + mean).clamp(0, 1)

In [ ]:
img = caches["train"]["images"][7]
geometries = {"crop": {"resize_mode": "crop", "img_size": 224},
              "squash": {"resize_mode": "squash", "img_size": 224},
              "rect": {"resize_mode": "rect", "img_size": [192, 288]}}
fig, ax = plt.subplots(len(geometries), 5, figsize=(18, 3.3 * len(geometries)))
for row, (name, g) in zip(ax, geometries.items()):
    row[0].imshow(denormalize(make_transforms(False, g)(img)).permute(1, 2, 0)); row[0].set_title(f"eval, {name}")
    t_aug = make_transforms(True, dict(g, aug="strong"))
    for k, a in enumerate(row[1:]):
        a.imshow(denormalize(t_aug(img)).permute(1, 2, 0)); a.set_title(f"train aug #{k + 1} ({name})")
    for a in row:
        a.axis("off")
plt.tight_layout()
plt.show()

> *Observations:* the center crop cuts the nose of the aircraft, squashing keeps it at the cost of a horizontal compression, while the rectangular input keeps the whole aircraft essentially undistorted. Too small random crops keep only a portion of the fuselage and lose discriminative parts, so the lower bound of the crop scale (`rrc_scale`) is treated as a hyperparameter.

## 2. Training utilities

Shared by all parts of the assignment:
- **2.1** optimizer, learning rate schedules (step, cosine with warm-up, one-cycle — L6), MixUp / CutMix;
- **2.2** training loop with mixed precision, model selection on the **validation** split only, checkpoints and history saved on
  Drive at every epoch, automatic resume after a disconnection;
- **2.3** `run_experiment`: runs an experiment only if it has not been completed yet, otherwise it just loads its results — the
  notebook can be executed top to bottom without retraining anything; the test split is evaluated once, on the best-validation
  checkpoint, at the end of each run;
- **2.4** plots and tables built from the runs saved on Drive (including the ones of the other team member).

### 2.1 Optimizers, schedules and batch mixing

In [ ]:
def build_optimizer(params, cfg: dict):
    """`params` is either model.parameters() or a list of parameter groups (e.g. different LRs in Part 2B)."""
    name, lr, wd = cfg.get("optimizer", "sgd"), cfg["lr"], cfg.get("wd", 0.0)
    if name == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=cfg.get("momentum", 0.9), nesterov=True, weight_decay=wd)
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    raise ValueError(name)


def build_scheduler(optimizer, cfg: dict, steps_per_epoch: int):
    """All schedules are updated at every step (mini-batch)."""
    name = cfg.get("scheduler", "cosine")
    total = cfg["epochs"] * steps_per_epoch
    warmup = cfg.get("warmup_epochs", 0) * steps_per_epoch
    if name == "onecycle":
        return torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=[g["lr"] for g in optimizer.param_groups],
                                                   total_steps=total)
    if name == "cosine":
        def f(step):
            if step < warmup:
                return (step + 1) / warmup
            return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))
    elif name == "step":
        milestones = [m * steps_per_epoch for m in cfg.get("milestones", [])]

        def f(step):
            if step < warmup:
                return (step + 1) / warmup
            return cfg.get("gamma", 0.1) ** sum(step >= m for m in milestones)
    elif name == "constant":
        def f(step):
            return 1.0
    else:
        raise ValueError(name)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, f)


def build_batch_mix(cfg: dict):
    """MixUp / CutMix on whole batches (L6). Returns None if both are disabled."""
    ops = []
    if cfg.get("mixup_alpha", 0) > 0:
        ops.append(v2.MixUp(alpha=cfg["mixup_alpha"], num_classes=num_classes))
    if cfg.get("cutmix_alpha", 0) > 0:
        ops.append(v2.CutMix(alpha=cfg["cutmix_alpha"], num_classes=num_classes))
    if not ops:
        return None
    return ops[0] if len(ops) == 1 else v2.RandomChoice(ops)

### 2.2 Training loop

The validation loss is always the plain cross-entropy (without label smoothing), so that it is comparable across runs.

In [ ]:
def save_json(obj, path):
    tmp = f"{path}.tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def load_json(path):
    with open(path) as f:
        return json.load(f)


@torch.no_grad()
def evaluate(model, loader, return_preds=False):
    model.eval()
    use_amp = device.type == "cuda"
    tot_loss, correct, n, preds, labels = 0.0, 0, 0, [], []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
            out = model(x)
        tot_loss += F.cross_entropy(out.float(), y, reduction="sum").item()
        p = out.argmax(1)
        correct += (p == y).sum().item()
        n += y.numel()
        if return_preds:
            preds.append(p.cpu()); labels.append(y.cpu())
    res = {"loss": tot_loss / n, "acc": correct / n}
    if return_preds:
        res["preds"], res["labels"] = torch.cat(preds), torch.cat(labels)
    return res


def train_model(model, train_loader, val_loader, cfg, run_dir, params=None):
    """Trains `model` and writes in `run_dir`: history.json (every epoch), best.pt (best validation accuracy),
    last.pt (full state, to resume after a disconnection)."""
    use_amp = cfg.get("amp", True) and device.type == "cuda"
    optimizer = build_optimizer(model.parameters() if params is None else params, cfg)
    scheduler = build_scheduler(optimizer, cfg, len(train_loader))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.get("label_smoothing", 0.0))
    batch_mix = build_batch_mix(cfg)

    history = {k: [] for k in ["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "lr", "epoch_time"]}
    start_epoch, best_acc, best_epoch = 0, -1.0, -1
    last_path = f"{run_dir}/last.pt"
    if os.path.exists(last_path):
        ckpt = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model"]); optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"]); scaler.load_state_dict(ckpt["scaler"])
        history, best_acc, best_epoch, start_epoch = ckpt["history"], ckpt["best_acc"], ckpt["best_epoch"], ckpt["epoch"] + 1
        print(f"Resuming from epoch {start_epoch} (best val acc so far {best_acc:.4f} at epoch {best_epoch})")

    pbar = tqdm(range(start_epoch, cfg["epochs"]), desc=os.path.basename(run_dir), initial=start_epoch, total=cfg["epochs"])
    for epoch in pbar:
        t0 = time.time()
        model.train()
        tot_loss, correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            y_hard = y
            if batch_mix is not None:
                x, y = batch_mix(x, y)  # y becomes a soft label
                y_hard = y.argmax(1)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                out = model(x)
                loss = criterion(out, y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if cfg.get("grad_clip", 0) > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            tot_loss += loss.item() * y_hard.numel()
            correct += (out.argmax(1) == y_hard).sum().item()
            n += y_hard.numel()

        val = evaluate(model, val_loader)
        for k, v in [("epoch", epoch), ("train_loss", tot_loss / n), ("train_acc", correct / n), ("val_loss", val["loss"]),
                     ("val_acc", val["acc"]), ("lr", optimizer.param_groups[0]["lr"]), ("epoch_time", time.time() - t0)]:
            history[k].append(v)
        if val["acc"] > best_acc:
            best_acc, best_epoch = val["acc"], epoch
            torch.save(model.state_dict(), f"{run_dir}/best.pt")
        save_json(history, f"{run_dir}/history.json")
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                    "scaler": scaler.state_dict(), "history": history, "best_acc": best_acc, "best_epoch": best_epoch,
                    "epoch": epoch}, last_path)
        pbar.set_postfix(train_acc=f"{correct / n:.3f}", val_acc=f"{val['acc']:.3f}", best=f"{best_acc:.3f}")
    return history, best_acc, best_epoch

### 2.3 Running an experiment

`run_experiment(name, build_model, cfg)` saves everything in `RUNS_DIR/<AUTHOR>_<name>/`. If the run has already been completed
it only loads its results; if it was interrupted it resumes it. If a run with the same name exists but with a *different*
configuration, it stops with an error, so that results are never silently mixed up: change the name instead.

In [ ]:
def run_dir_of(name, author=None):
    return f"{RUNS_DIR}/{author or AUTHOR}_{name}"


def run_experiment(name, build_model, cfg, build_params=None, force=False):
    """build_model(cfg) -> nn.Module; build_params(model, cfg) -> parameter groups (optional)."""
    run_dir = run_dir_of(name)
    cfg = json.loads(json.dumps(cfg))  # normalize (tuples -> lists) to compare it with the saved one
    if os.path.exists(f"{run_dir}/config.json"):
        saved = {k: v for k, v in load_json(f"{run_dir}/config.json").items() if k in cfg}
        if saved != cfg and not force:
            raise ValueError(f"'{name}' already exists with a different config: use another name (or force=True to delete the old run).")
    if force and os.path.exists(run_dir):
        shutil.rmtree(run_dir)  # start from scratch: never resume a checkpoint of a different config
    if os.path.exists(f"{run_dir}/results.json"):
        res = load_json(f"{run_dir}/results.json")
        print(f"[{name}] already completed: val acc {res['best_val_acc']:.4f} | test acc {res['test_acc']:.4f}")
        return res

    os.makedirs(run_dir, exist_ok=True)
    fix_random(cfg.get("seed", SEED))
    model = build_model(cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    save_json(dict(cfg, author=AUTHOR, gpu=GPU_NAME, n_params=n_params, n_trainable=n_trainable),
              f"{run_dir}/config.json")
    print(f"[{name}] {n_params / 1e6:.2f}M parameters ({n_trainable / 1e6:.2f}M trainable) on {GPU_NAME}")

    train_loader = make_loader("train", cfg, train=True, seed=cfg.get("seed", SEED))
    val_loader = make_loader("val", cfg, train=False, batch_size=128)
    params = build_params(model, cfg) if build_params else None
    t0 = time.time()
    history, best_acc, best_epoch = train_model(model, train_loader, val_loader, cfg, run_dir, params)

    # The test split is used only here, once, on the checkpoint selected on the validation split
    model.load_state_dict(torch.load(f"{run_dir}/best.pt", map_location=device))
    test = evaluate(model, make_loader("test", cfg, train=False, batch_size=128))
    res = {"best_epoch": best_epoch, "best_val_acc": best_acc, "test_acc": test["acc"], "test_loss": test["loss"],
           "train_acc_at_best": history["train_acc"][best_epoch], "n_params": n_params,
           "minutes": sum(history["epoch_time"]) / 60}
    save_json(res, f"{run_dir}/results.json")
    print(f"[{name}] best epoch {best_epoch} | val acc {best_acc:.4f} | test acc {test['acc']:.4f}")
    return res


def load_trained_model(name, build_model, author=None):
    run_dir = run_dir_of(name, author)
    cfg = load_json(f"{run_dir}/config.json")
    model = build_model(cfg).to(device)
    model.load_state_dict(torch.load(f"{run_dir}/best.pt", map_location=device))
    return model.eval(), cfg

### 2.4 Plots and tables from saved runs

In [ ]:
import pandas as pd


def list_runs(prefix=""):
    """Full names of all the runs saved on Drive (both authors), optionally filtered by prefix."""
    return sorted(d for d in os.listdir(RUNS_DIR) if d.startswith(prefix) and os.path.exists(f"{RUNS_DIR}/{d}/history.json"))


def _load(full_name):
    d = f"{RUNS_DIR}/{full_name}"
    res = load_json(f"{d}/results.json") if os.path.exists(f"{d}/results.json") else {}
    return load_json(f"{d}/config.json"), load_json(f"{d}/history.json"), res


def plot_runs(full_names, labels=None, title=None):
    """Loss, accuracy (dashed = train, solid = validation) and learning rate of some runs."""
    labels = labels or full_names
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
    for full_name, label in zip(full_names, labels):
        _, h, _ = _load(full_name)
        line, = ax[0].plot(h["epoch"], h["val_loss"], label=label)
        c = line.get_color()
        ax[0].plot(h["epoch"], h["train_loss"], "--", color=c, alpha=0.6)
        ax[1].plot(h["epoch"], h["val_acc"], color=c, label=label)
        ax[1].plot(h["epoch"], h["train_acc"], "--", color=c, alpha=0.6)
        ax[2].plot(h["epoch"], h["lr"], color=c, label=label)
    ax[0].set_title("loss"); ax[1].set_title("accuracy"); ax[2].set_title("learning rate")
    for a in ax:
        a.set_xlabel("epoch"); a.grid(alpha=0.3)
    ax[1].legend(fontsize=8)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def results_table(full_names, labels=None, reference=None):
    """One row per run; `reference` (a full name) adds the differences in accuracy w.r.t. that run."""
    labels = labels or full_names
    rows = []
    for full_name, label in zip(full_names, labels):
        cfg, h, res = _load(full_name)
        rows.append({"run": label, "params (M)": cfg.get("n_params", float("nan")) / 1e6,
                     "epochs": len(h["epoch"]), "best epoch": res.get("best_epoch"),
                     "train acc @best": res.get("train_acc_at_best"), "val acc": res.get("best_val_acc"),
                     "test acc": res.get("test_acc"), "minutes": res.get("minutes"), "gpu": cfg.get("gpu")})
    df = pd.DataFrame(rows).set_index("run")
    if reference is not None:  # decisions are taken on the validation accuracy; Δ test is only reported
        ref = _load(reference)[2]
        df["Δ val acc"] = df["val acc"] - ref.get("best_val_acc")
        df["Δ test acc"] = df["test acc"] - ref.get("test_acc")
    return df.style.format(precision=3, na_rep="-")

### 2.5 Smoke test

A tiny network trained for 2 epochs on small images, only to check the whole chain (loaders, training, saving on Drive,
test evaluation, tables and plots). It runs on CPU as well; its folder is deleted at the end.

In [ ]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    def tiny_net(cfg):
        return nn.Sequential(nn.Conv2d(3, 16, 3, stride=2, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
                             nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, num_classes))

    smoke_cfg = {"img_size": [64, 96], "resize_mode": "rect", "aug": "strong", "batch_size": 64, "epochs": 2,
                 "optimizer": "sgd", "lr": 0.05, "wd": 5e-4, "scheduler": "cosine", "warmup_epochs": 1,
                 "label_smoothing": 0.1, "mixup_alpha": 0.2, "cutmix_alpha": 1.0}
    run_experiment("smoke_test", tiny_net, smoke_cfg)
    run_experiment("smoke_test", tiny_net, smoke_cfg)   # second call: must only load the results
    display(results_table([f"{AUTHOR}_smoke_test"]))
    plot_runs([f"{AUTHOR}_smoke_test"])
    shutil.rmtree(run_dir_of("smoke_test"))

## Part 1: design your own network

Your goal is to implement a convolutional neural network for image classification and train it from scratch on `FGVCAircraft`. You should consider yourselves satisfied once you obtain a classification accuracy on the test split of ~50%. You are free to achieve this however you want, except for a few rules you must follow:

- Compile this notebook by displaying the results obtained by the best model you found throughout your experimentation; then show how, by removing some of its components, its performance drops. In other words, do an *ablation study* to prove that your design choices have a positive impact on the final result.

- Do not instantiate an off-the-self PyTorch network. Instead, construct your network as a composition of existing PyTorch layers. In more concrete terms, you can use e.g. `torch.nn.Linear`, but you cannot use e.g. `torchvision.models.alexnet`.

- Show your results and ablations with plots, tables, images, etc. — the clearer, the better.

Don't be too concerned with your model performance: the ~50% is just to give you an idea of when to stop. Keep in mind that a thoroughly justified model with lower accuracy will be rewarded more points than a poorly experimentally validated model with higher accuracy.

### P1.1 Network design

All the networks of Part 1 belong to one configurable family, `AircraftNet`, so that every ablation is obtained by changing
a single key of the configuration:

- **stem**: a 3×3 convolution with stride 2 that halves the resolution early (images are large, fine details are in the
  following layers);
- **4 stages** with increasing width (`widths`); the first block of every stage halves the resolution with stride 2, so the
  receptive field grows while the cost per stage stays similar. `depths` sets the number of blocks per stage;
- **blocks** (`block`): `"plain"` = two 3×3 conv–BN–ReLU; `"residual"` = the same two convolutions plus a skip connection
  (a 1×1 convolution when the shape changes), as in ResNet (L5);
- **batch normalization** after every convolution, which can be switched off (`bn`);
- **head**: global average pooling → dropout (`dropout`) → linear classifier. Global average pooling makes the network
  independent of the input size (needed for the rectangular input) and avoids a large, overfitting-prone fully connected layer.

Weights are initialized with the Kaiming (He) initialization, designed for ReLU networks (L4).

In [ ]:
class Block(nn.Module):
    """Two 3x3 convolutions (+ BN + ReLU); with residual=True, a skip connection is added before the last ReLU."""

    def __init__(self, c_in, c_out, stride, residual, bn):
        super().__init__()
        self.residual = residual
        norm = (lambda c: nn.BatchNorm2d(c)) if bn else (lambda c: nn.Identity())
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=not bn)
        self.bn1 = norm(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1, bias=not bn)
        self.bn2 = norm(c_out)
        self.shortcut = None
        if residual and (stride != 1 or c_in != c_out):
            self.shortcut = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride=stride, bias=not bn), norm(c_out))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.residual:
            out = out + (x if self.shortcut is None else self.shortcut(x))
        return F.relu(out)


class AircraftNet(nn.Module):
    def __init__(self, num_classes, widths=(32, 64, 128, 256), depths=(1, 1, 1, 1), stem_width=32,
                 block="plain", bn=True, dropout=0.0):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, stem_width, 3, stride=2, padding=1, bias=not bn),
                                  nn.BatchNorm2d(stem_width) if bn else nn.Identity(), nn.ReLU(inplace=True))
        layers, c_in = [], stem_width
        for c_out, depth in zip(widths, depths):
            for i in range(depth):
                layers.append(Block(c_in, c_out, stride=2 if i == 0 else 1, residual=(block == "residual"), bn=bn))
                c_in = c_out
        self.stages = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(c_in, num_classes))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.head(self.stages(self.stem(x)))


def build_aircraftnet(cfg):
    return AircraftNet(num_classes, widths=tuple(cfg.get("widths", (32, 64, 128, 256))),
                       depths=tuple(cfg.get("depths", (1, 1, 1, 1))), stem_width=cfg.get("stem_width", 32),
                       block=cfg.get("block", "plain"), bn=cfg.get("bn", True), dropout=cfg.get("dropout", 0.0))

### P1.2 Baseline configuration and sanity checks

Following the recipe of L6 (slide 40), we start from a deliberately simple baseline — plain blocks, basic augmentation,
no label smoothing, no MixUp/CutMix, no dropout — and first verify that the pipeline is correct:
1. with random weights the network should predict an almost uniform distribution, so the initial loss should be close to
   $\ln(100) \approx 4.61$;
2. the network must be able to reach (almost) zero loss on a **single batch**: if it cannot, there is a bug in the data,
   the labels or the model.

In [ ]:
P1_BASE = {
    "seed": SEED,
    # input
    "img_size": [192, 288], "resize_mode": "rect", "aug": "basic", "rrc_scale": [0.35, 1.0],
    # model
    "widths": [32, 64, 128, 256], "depths": [1, 1, 1, 1], "stem_width": 32, "block": "plain", "bn": True, "dropout": 0.0,
    # training
    "batch_size": 64, "epochs": 40, "optimizer": "sgd", "lr": 0.05, "wd": 5e-4,
    "scheduler": "cosine", "warmup_epochs": 2, "label_smoothing": 0.0, "mixup_alpha": 0.0, "cutmix_alpha": 0.0,
}

fix_random(SEED)
model = build_aircraftnet(P1_BASE).to(device)
x, y = next(iter(make_loader("train", dict(P1_BASE, aug="none"), train=True, batch_size=32)))
x, y = x.to(device), y.to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M | input {tuple(x.shape)} -> output {tuple(model(x).shape)}")

# 1) initial loss
model.eval()
with torch.no_grad():
    print(f"initial loss: {F.cross_entropy(model(x), y).item():.3f} (expected ~{math.log(num_classes):.3f})")

# 2) overfit a single batch
model.train()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
for step in range(200):
    loss = F.cross_entropy(model(x), y)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
plt.figure(figsize=(5, 3))
plt.plot(losses); plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss")
plt.title(f"Overfitting one batch of 32 images (final loss {losses[-1]:.4f})")
plt.show()
del model, opt

> *Observations:* the initial loss matches $\ln(100)$ and the network drives the loss on a single batch to ~0.003 in 200 steps: data, labels and model are consistent, so the pipeline can be trusted.

### P1.3 Baseline

First real training run, which gives a first reference accuracy for the following experiments.

In [ ]:
RUN_P1_BASELINE = True

if RUN_P1_BASELINE:
    run_experiment("p1_baseline", build_aircraftnet, P1_BASE)

name = f"{AUTHOR}_p1_baseline"
display(results_table([name]))
plot_runs([name])

> *Observations:* the baseline reaches 34.5% validation accuracy. Three issues emerge:
> 1. **large generalization gap**: 65% training vs 35% validation accuracy at the best epoch;
> 2. **limited capacity**: even on the training set (with augmentation) the network only reaches 65%, so it is not able to fit the data;
> 3. **instability at high learning rate**: while the LR is above ~0.02, the validation loss shows large spikes (epochs 2, 9, 11, 19) although the training loss decreases smoothly; the curves become stable only when the cosine schedule lowers the LR.
>
> Following the recipe, we first increase the capacity of the network until it can fit the training set, then we regularize it.

### P1.4 Increasing the capacity

One change at a time with respect to the baseline, everything else fixed (same 40 epochs, same optimizer and schedule):
- `p1_res`: residual blocks instead of plain blocks — skip connections ease the optimization (L5) and may reduce the
  instability observed above;
- `p1_res_d2`: residual blocks, 2 blocks per stage (twice as deep);
- `p1_res_d2_w2`: residual, 2 blocks per stage, twice as wide (64 → 512 channels).

In [ ]:
RUN_P1_CAPACITY = True

P1_CAPACITY = {
    "p1_res":       dict(P1_BASE, block="residual"),
    "p1_res_d2":    dict(P1_BASE, block="residual", depths=[2, 2, 2, 2]),
    "p1_res_d2_w2": dict(P1_BASE, block="residual", depths=[2, 2, 2, 2], widths=[64, 128, 256, 512]),
}
if RUN_P1_CAPACITY:
    for run_name, cfg in P1_CAPACITY.items():
        run_experiment(run_name, build_aircraftnet, cfg)

names = [f"{AUTHOR}_p1_baseline"] + [f"{AUTHOR}_{n}" for n in P1_CAPACITY]
display(results_table(names, reference=f"{AUTHOR}_p1_baseline"))
plot_runs(names)

> *Observations:* model selection is based on the validation accuracy (single seed, differences of 1–2 points are within noise).
> - **Residual blocks alone hurt** (`p1_res`, −8 points). With one block per stage, every block downsamples, so every skip connection is a strided 1×1 projection and never an identity: the network gets none of the benefits of residual learning, while the 1×1 stride-2 path only sees one pixel out of four. Skip connections pay off with deeper stages (`p1_res_d2`), where most shortcuts are identities.
> - **Depth helps**: `p1_res_d2` is the best model (39.2%), with training accuracy rising from 65% to 74%.
> - **Width increases the capacity but not the accuracy**: `p1_res_d2_w2` (11.2M parameters) fits the training set much better (84%), but validation accuracy does not improve (38.0%): capacity is no longer the bottleneck, and the generalization gap (46 points) is now the main problem.
> - **Instability persists in all models** while the LR is high (validation loss spikes up to 7–8), so the learning rate of 0.05 is likely too large; the curves become smooth only as the cosine schedule decays.
>
> Next step: regularization, on both the deeper and the wider network (the larger model may benefit more from it).

### P1.5 Regularization

The networks now have enough capacity; the goal is to reduce the generalization gap (L6). Regularizers are first added
together to see how far they can go — their individual contribution is measured later, in the ablation study:
- `aug="strong"`: color jitter and random grayscale on top of random crops and flips;
- `label_smoothing=0.1`: targets are 0.9 for the correct class instead of 1, which discourages over-confident predictions;
- `dropout=0.3` before the classifier.

The same recipe is applied to the deeper (`d2`) and the wider (`d2_w2`) network; a third run uses a smaller learning rate
(0.02), to check whether it removes the instability observed so far.

In [ ]:
RUN_P1_REG = True

REG = {"aug": "strong", "label_smoothing": 0.1, "dropout": 0.3}
P1_REG = {
    "p1_d2_reg":         dict(P1_CAPACITY["p1_res_d2"], **REG),
    "p1_d2w2_reg":       dict(P1_CAPACITY["p1_res_d2_w2"], **REG),
    "p1_d2w2_reg_lr002": dict(P1_CAPACITY["p1_res_d2_w2"], **REG, lr=0.02),
}
if RUN_P1_REG:
    for run_name, cfg in P1_REG.items():
        run_experiment(run_name, build_aircraftnet, cfg)

names = [f"{AUTHOR}_p1_res_d2", f"{AUTHOR}_p1_res_d2_w2"] + [f"{AUTHOR}_{n}" for n in P1_REG]
display(results_table(names, reference=f"{AUTHOR}_p1_res_d2"))
plot_runs(names)

> *Observations:*
> - **Regularization pays off only with enough capacity.** On the wide network (`p1_d2w2_reg`, 11.2M) it gives the best model so far (39.9%) and shrinks the generalization gap from 46 to ~18 points; on the smaller network (`p1_d2_reg`, 2.8M) it causes underfitting (training accuracy 42%) and lowers validation accuracy by 7 points.
> - **The instability disappears with regularization**: the wide regularized model has no validation-loss spikes even at LR 0.05. A plausible explanation is label smoothing, which prevents over-confident logits, so wrong predictions no longer produce huge losses.
> - **A smaller learning rate is not the solution**: with LR 0.02 the network learns more slowly and loses 11 points after 40 epochs.
> - **All regularized models are still improving at the end of training** (training accuracy < 60% and validation accuracy still rising), so 40 epochs are too few: the next step is a longer schedule.

### P1.6 Longer training and MixUp / CutMix

Regularized models learn more slowly, so we train for 100 epochs instead of 40:
- `p1_d2w2_reg_e100`: the best model so far, trained 2.5× longer;
- `p1_d2_reg_e100`: the smaller network, to check whether it catches up with a longer schedule;
- `p1_d2w2_reg_mix_e100`: the best model with MixUp (α = 0.2) or CutMix (α = 1), chosen at random for every batch (L6):
  a stronger regularizer that also discourages relying on a single region of the image (e.g. the livery on the tail).

In [ ]:
RUN_P1_LONG = True

P1_LONG = {
    "p1_d2w2_reg_e100":     dict(P1_REG["p1_d2w2_reg"], epochs=100),
    "p1_d2_reg_e100":       dict(P1_REG["p1_d2_reg"], epochs=100),
    "p1_d2w2_reg_mix_e100": dict(P1_REG["p1_d2w2_reg"], epochs=100, mixup_alpha=0.2, cutmix_alpha=1.0),
}
if RUN_P1_LONG:
    for run_name, cfg in P1_LONG.items():
        run_experiment(run_name, build_aircraftnet, cfg)

names = [f"{AUTHOR}_p1_d2w2_reg"] + [f"{AUTHOR}_{n}" for n in P1_LONG]
display(results_table(names, reference=f"{AUTHOR}_p1_d2w2_reg"))
plot_runs(names)

> *Observations:*
> - **Training longer is by far the most effective change**: with 100 epochs the wide regularized network reaches **59.0% validation / 58.4% test accuracy** (+19 points w.r.t. 40 epochs), above the 50% target. Its training accuracy reaches 99%, yet validation accuracy keeps improving until epoch ~90: regularization lets the network fit the training set without memorizing it.
> - **The smaller network catches up** (56.5%, with 4× fewer parameters): the extra width is worth ~2.5 points.
> - **MixUp / CutMix do not help within 100 epochs** (57.8%, −1.3 points, within noise): training accuracy stays at ~70%, i.e. such a strong regularizer would need a longer schedule.
>
> We therefore fix `p1_d2w2_reg_e100` as the final model of Part 1.

### P1.7 First final model and ablation study

**First final model** (`p1_d2w2_reg_e100`): `AircraftNet` with residual blocks, 2 blocks per stage, widths 64–128–256–512
(11.2M parameters), batch normalization, dropout 0.3 before the classifier; rectangular 192×288 input with strong
augmentation; SGD with Nesterov momentum (LR 0.05, weight decay 5·10⁻⁴), cosine schedule with 2 warm-up epochs,
label smoothing 0.1, 100 epochs, batch size 64.

**Ablation study**: every run removes (or replaces) **one** component of the final model, everything else unchanged.
Some runs of the previous sections already differ from the final model in exactly one component and are reused.
A second run of the final model with a different seed estimates the run-to-run noise: differences smaller than that
should not be considered significant.

In [ ]:
def ablation_report(final_name, ablation, reused, title="Ablation study"):
    """Table + bar chart of the Δ validation accuracy of every ablation w.r.t. the final model.
    ablation: {run name: (description, config)}; reused: {run name: description}. Only completed runs are shown."""
    done = lambda n: os.path.exists(f"{run_dir_of(n)}/results.json")
    rows = [(final_name, "final model")] + [(n, d) for n, (d, _) in ablation.items() if done(n)] + \
           [(n, d) for n, d in reused.items() if done(n)]
    names, labels = [f"{AUTHOR}_{n}" for n, _ in rows], [d for _, d in rows]
    display(results_table(names, labels, reference=names[0]))
    ref = _load(names[0])[2]["best_val_acc"]
    deltas = [(_load(n)[2]["best_val_acc"] - ref) * 100 for n in names[1:]]
    order = np.argsort(deltas)
    plt.figure(figsize=(8, 0.45 * len(deltas) + 1))
    plt.barh([labels[1:][i] for i in order], [deltas[i] for i in order],
             color=["tab:red" if deltas[i] < 0 else "tab:green" for i in order])
    plt.axvline(0, color="black", lw=0.8)
    plt.xlabel("Δ validation accuracy w.r.t. the final model (points)")
    plt.title(title)
    plt.tight_layout()
    plt.show()


P1_FINAL_NAME = "p1_d2w2_reg_e100"
P1_FINAL = P1_LONG[P1_FINAL_NAME]

RUN_P1_ABLATION = True

P1_ABLATION = {  # run name: (description, config); ordered by expected informativeness
    "p1_abl_seed1":    ("final model, different seed",       dict(P1_FINAL, seed=SEED + 1)),
    "p1_abl_noaug":    ("no data augmentation",              dict(P1_FINAL, aug="none")),
    "p1_abl_plain":    ("plain blocks (no skip connections)", dict(P1_FINAL, block="plain")),
    "p1_abl_nobn":     ("no batch normalization",            dict(P1_FINAL, bn=False)),
    "p1_abl_nols":     ("no label smoothing",                dict(P1_FINAL, label_smoothing=0.0)),
    "p1_abl_nodrop":   ("no dropout",                        dict(P1_FINAL, dropout=0.0)),
    "p1_abl_crop":     ("square center crop, 224×224",       dict(P1_FINAL, resize_mode="crop", img_size=224)),
}
P1_REUSED = {  # runs of the previous sections that differ from the final model in one component only
    "p1_d2_reg_e100":       "half width (2.8M parameters)",
    "p1_d2w2_reg":          "40 epochs instead of 100",
    "p1_d2w2_reg_mix_e100": "+ MixUp / CutMix",
}

if RUN_P1_ABLATION:
    for run_name, (_, cfg) in P1_ABLATION.items():
        run_experiment(run_name, build_aircraftnet, cfg)

In [ ]:
ablation_report(P1_FINAL_NAME, P1_ABLATION, P1_REUSED, title="Ablation study of the first final model")

> *Observations:* a second run of the final model with a different seed gives 59.1% vs 59.0% validation accuracy (1.3 points apart on the test set): differences below ~1.5 points are within noise.
> - **Essential components**: data augmentation (−30.7 points: without it the network memorizes the training set, 100% training accuracy), the length of training (−19.1 with 40 epochs) and batch normalization (−13.9: without it even the training accuracy stops at 76%, so its main role here is to make optimization possible).
> - **Small contributions, at the level of noise**: half width (−2.5), label smoothing (−1.8), dropout (−1.7); adding MixUp/CutMix (−1.3) does not help.
> - **Skip connections are not needed** (+1.0, within noise): with batch normalization, a 16-layer plain network is trained just as well; residual connections become important for much deeper networks.
> - **The square center crop is better than the rectangular input (+3.3 points).** This contradicts our hypothesis, but the comparison mixes two factors: in crop mode the image is resized to a height of 256 px before cropping, in rect mode to 192 px, so the crop sees the aircraft at 1.33× higher resolution. It loses nose and tail, but it gains details on fuselage, engines and windows, which matter more in a fine-grained task. The next section separates the two factors.

### P1.8 Geometry or resolution?

`p1_rect256` uses the rectangular input at 256×384: the same scale as the center crop, but with the whole aircraft.
- if it beats the crop, keeping the whole aircraft *does* help and the first rectangular model was only too low-resolution;
- if it does not, what matters is the resolution on the central part of the aircraft.

The best of the three input geometries on the **validation** set becomes the final model of Part 1.

In [ ]:
RUN_P1_GEOMETRY = True

P1_GEOMETRY = {  # run name: (description, config) — all identical except for the input geometry
    "p1_d2w2_reg_e100": ("rectangular 192×288 (first final model)", P1_FINAL),
    "p1_abl_crop":      ("square center crop 224×224",              P1_ABLATION["p1_abl_crop"][1]),
    "p1_rect256":       ("rectangular 256×384",                     dict(P1_FINAL, img_size=[256, 384])),
}
if RUN_P1_GEOMETRY:
    run_experiment("p1_rect256", build_aircraftnet, P1_GEOMETRY["p1_rect256"][1])

names = [f"{AUTHOR}_{n}" for n in P1_GEOMETRY]
display(results_table(names, [d for d, _ in P1_GEOMETRY.values()], reference=names[0]))
plot_runs(names, [d for d, _ in P1_GEOMETRY.values()])

# Model selection on the validation accuracy only
P1_FINAL2_NAME = max(P1_GEOMETRY, key=lambda n: _load(f"{AUTHOR}_{n}")[2]["best_val_acc"])
P1_FINAL2 = P1_GEOMETRY[P1_FINAL2_NAME][1]
print(f"Final model of Part 1: {P1_FINAL2_NAME} ({P1_GEOMETRY[P1_FINAL2_NAME][0]})")

> *Observations:* TODO

### P1.9 Final model and ablation study

The ablation study is repeated with respect to the final model selected in P1.8, with the same components as in P1.7
plus the width, the length of training and MixUp/CutMix, which are now re-trained with the final input geometry.
The other two input geometries of P1.8 differ from the final model in one component only and are reused.

In [ ]:
RUN_P1_ABLATION2 = True

F2 = P1_FINAL2
P1_ABLATION2 = {  # ordered by expected informativeness
    "p1_abl2_seed1":     ("final model, different seed",        dict(F2, seed=SEED + 1)),
    "p1_abl2_noaug":     ("no data augmentation",               dict(F2, aug="none")),
    "p1_abl2_plain":     ("plain blocks (no skip connections)", dict(F2, block="plain")),
    "p1_abl2_nobn":      ("no batch normalization",             dict(F2, bn=False)),
    "p1_abl2_nols":      ("no label smoothing",                 dict(F2, label_smoothing=0.0)),
    "p1_abl2_nodrop":    ("no dropout",                         dict(F2, dropout=0.0)),
    "p1_abl2_halfwidth": ("half width (2.8M parameters)",       dict(F2, widths=[32, 64, 128, 256])),
    "p1_abl2_e40":       ("40 epochs instead of 100",           dict(F2, epochs=40)),
    "p1_abl2_mix":       ("+ MixUp / CutMix",                   dict(F2, mixup_alpha=0.2, cutmix_alpha=1.0)),
}
P1_REUSED2 = {n: d for n, (d, _) in P1_GEOMETRY.items() if n != P1_FINAL2_NAME}

if RUN_P1_ABLATION2:
    for run_name, (_, cfg) in P1_ABLATION2.items():
        run_experiment(run_name, build_aircraftnet, cfg)

In [ ]:
ablation_report(P1_FINAL2_NAME, P1_ABLATION2, P1_REUSED2, title="Ablation study of the final model")

> *Observations:* TODO

## Part 2: fine-tune an existing network

Your goal is to fine-tune a pretrained ResNet-18 model on `FGVCAircraft`. Use the implementation provided by PyTorch, i.e. the opposite of part 1. Specifically, use the PyTorch ResNet-18 model pretrained on ImageNet-1K (V1). Divide your fine-tuning into two parts:

2A. First, fine-tune the ResNet-18 with the same training hyperparameters you used for your best model in part 1.

2B. Then, tweak the training hyperparameters to increase the accuracy on the test split. Justify your choices by analyzing the training plots and/or citing sources that guided you in your decisions — papers, blog posts, YouTube videos, or whatever else you may find useful. You should consider yourselves satisfied once you obtain a classification accuracy on the test split of ~70%.